# 项目：评估和清理英国电商公司销售数据

## 分析目标

此数据分析的目的是，根据市场销售数据，挖掘畅销产品，以便制定更有效的市场策略来提升营收。

本实战项目的目的在于练习评估数据干净和整洁度，并且基于评估结果，对数据进行清洗，从而得到可供下一步分析的数据。

## 简介

原始数据集记录了一家英国在线零售公司在2010年12月1日至2011年12月9日期间的所有交易情况，涵盖了该公司在全球不同国家和地区的业务数据。该公司主要销售覆盖各个场景的礼品，包括但不限于生日礼品、结婚纪念品、圣诞礼品等等。该公司的客户群体主要包括批发商和个人消费者，其中批发商占据了相当大的比例。

数据每列的含义如下：
- `InvoiceNo`: 发票号码。6位数，作为交易的唯一标识符。如果这个代码以字母“c”开头，表示这笔交易被取消。
- `StockCode`: 产品代码。5位数，作为产品的唯一标识符。
- `Description`: 产品名称。
- `Quantity`: 产品在交易中的数量。
- `InvoiceDate`: 发票日期和时间。交易发生的日期和时间。
- `UnitPrice`: 单价。价格单位为英镑（£）。
- `CustomerID`: 客户编号。5位数，作为客户的唯一标识符。
- `Country`: 国家名称。客户所居住的国家的名称。

## Access

导入数据分析所需库 `pandas` 并读取原始数据文件，解析为DataFrame，赋值给变量`original_df`，然后利用`.info()`，`.sample()`方法初步评估

In [6]:
import pandas as pd

In [4]:
original_df = pd.read_csv("e_commerce.csv")

In [10]:
original_df.info() # InvoiceDate, CustomerID 数据类型有误

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [8]:
original_df.sample(10) # 存在缺失值 [CustomerID] 和异常值 [StockCode]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
204100,554630,22649,STRAWBERRY FAIRY CAKE TEAPOT,2,5/25/2011 11:34,4.95,13268.0,United Kingdom
305678,563707,22627,MINT KITCHEN SCALES,4,8/18/2011 14:53,8.50,16626.0,United Kingdom
104549,545186,84659A,WHITE TRAVEL ALARM CLOCK,2,2/28/2011 15:05,2.55,17841.0,United Kingdom
529911,580757,82482,WOODEN PICTURE FRAME WHITE FINISH,6,12/6/2011 10:27,2.95,13709.0,United Kingdom
93492,544295,82580,BATHROOM METAL SIGN,24,2/17/2011 12:43,0.42,14298.0,United Kingdom
277054,561087,23296,SET OF 6 TEA TIME BAKING CASES,24,7/25/2011 9:48,1.25,15144.0,United Kingdom
188000,553013,22666,RECIPE BOX PANTRY YELLOW DESIGN,1,5/12/2011 18:19,5.79,NaN,United Kingdom
440455,574532,71101E,STANDING FAIRY POLE SUPPORT,12,11/4/2011 14:19,0.85,15676.0,United Kingdom
385697,570211,22996,TRAVEL CARD WALLET VINTAGE TICKET,2,10/9/2011 11:15,0.42,13558.0,United Kingdom
18701,537823,21884,CAKES AND BOWS GIFT TAPE,1,12/8/2010 14:25,1.66,NaN,United Kingdom


## Evaluation

评估数据时，结构上：每列是一个变量，每行是一个观察值，每个单元格是一个值（根据 `.sample()` 已知数据不存在结构性问题）。
内容上：找出缺失数据 `.isnull` ，重复数据 `.duplicated` ，不一致数据和无效数据

### Evaluating Missing Data

In [19]:
original_df.isnull().sum() # Description, CustomerID存在数据缺失

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [12]:
original_df[original_df["Description"].isnull()]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/2010 11:52,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,12/1/2010 14:32,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,12/1/2010 14:33,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,12/1/2010 14:33,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,12/1/2010 14:34,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,12/7/2011 18:26,0.0,NaN,United Kingdom
535326,581203,23406,NaN,15,12/7/2011 18:31,0.0,NaN,United Kingdom
535332,581209,21620,NaN,6,12/7/2011 18:35,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,12/8/2011 10:33,0.0,NaN,United Kingdom


从输出结果，缺失 `Description` 的交易数据，`UnitPrice`都为0，为验证这一点，继续增加筛选条件

In [14]:
original_df[(original_df["Description"].isnull()) & (original_df["UnitPrice"] != 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


输出结果符合猜想，`Description` 和 `UnitPrice` 都是后续数据分析的重要变量，同时缺失则证明对应行数据可以被删除

In [17]:
original_df[original_df["CustomerID"].isnull()] # 仅缺失 CustomerID 不影响后续数据分析

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/2010 11:52,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,12/1/2010 14:32,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/1/2010 14:32,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,12/1/2010 14:32,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,12/1/2010 14:32,1.66,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
541536,581498,85099B,JUMBO BAG RED RETROSPOT,5,12/9/2011 10:26,4.13,NaN,United Kingdom
541537,581498,85099C,JUMBO BAG BAROQUE BLACK WHITE,4,12/9/2011 10:26,4.13,NaN,United Kingdom
541538,581498,85150,LADIES & GENTLEMEN METAL SIGN,1,12/9/2011 10:26,4.96,NaN,United Kingdom
541539,581498,85174,S/4 CACTI CANDLES,1,12/9/2011 10:26,10.79,NaN,United Kingdom


### Evaluating Duplicate Data

根据数据变量的含义来看，虽然`InvoiceNo`, `StockCode`, `CustomerID`都是唯一标识符，但一次交易可能包含多种商品，因此 `InvoiceNo` 允许重复； 不同交易可能包含同一件商品，因此 `StockCode` 允许重复。顾客可以多次交易或下单多个商品， 因此 `CustomerID` 允许重复

### Evaluating Inconsistent Data

In [29]:
original_df["Country"].value_counts() # 后续需要统一 USA|United States, UK|U.K.|United Kingdom

Country
United Kingdom          495266
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
China                      288
Singapore                  229
USA                        218
UK                         211
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United States               73
United Arab Emirates        68


In [31]:
original_df.describe() # 出现负数，需要评估其是否具有意义

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [33]:
original_df[(original_df["Quantity"] < 0) & (original_df["InvoiceNo"].str[0] != "C")]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,12/1/2010 16:50,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,12/2/2010 14:42,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,12/3/2010 15:30,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,12/3/2010 15:30,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,12/3/2010 15:30,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,12/7/2011 18:36,0.0,NaN,United Kingdom
535335,581212,22578,lost,-1050,12/7/2011 18:38,0.0,NaN,United Kingdom
535336,581213,22576,check,-30,12/7/2011 18:38,0.0,NaN,United Kingdom
536908,581226,23090,missing,-338,12/8/2011 9:56,0.0,NaN,United Kingdom


In [34]:
original_df[(original_df["Quantity"] < 0) & (original_df["InvoiceNo"].str[0] != "C") & (original_df["UnitPrice"] != 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


后续清理步骤中，应该删除 `["Quantity"] < 0` ，因为这意味着要么订单已经取消，要么单价为0，没有参考价值

In [35]:
original_df[original_df["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,8/12/2011 14:51,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,8/12/2011 14:52,-11062.06,NaN,United Kingdom


后续清理步骤中，应该删除 `["UnitPrice"] < 0`

## Cleaning

In [37]:
cleaned_df = original_df.copy()
cleaned_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [46]:
# 清理缺失数据

cleaned_df.dropna(subset=["Description", "UnitPrice"], inplace=True)
cleaned_df.isnull().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     133626
Country             0
dtype: int64

In [53]:
# 清理不一致数据

cleaned_df["Country"] = cleaned_df["Country"].replace({"USA": "United States", "U.K.": "United Kingdom", "UK": "United Kingdom"})

In [54]:
l1 = len(cleaned_df[cleaned_df["Country"] == "USA"])
l2 = len(cleaned_df[cleaned_df["Country"] == "U.K."])
l3 = len(cleaned_df[cleaned_df["Country"] == "UK"])
print(l1, l2, l3, sep=" ")

0 0 0


In [41]:
# 转换数据类型

cleaned_df["InvoiceDate"] = pd.to_datetime(cleaned_df["InvoiceDate"])
cleaned_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [49]:
def fmt_id(x):
    if pd.isna(x):
        return pd.NA
    return f"{str(int(x))}"

In [50]:
cleaned_df["CustomerID"] = cleaned_df["CustomerID"].apply(fmt_id)
cleaned_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [16]:
# 清理重复数据

df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [55]:
# 清理异常数据

cleaned_df = cleaned_df[(cleaned_df["Quantity"] >= 0) & (cleaned_df["UnitPrice"] >= 0)]

In [56]:
cleaned_df[cleaned_df["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [57]:
cleaned_df[cleaned_df["Quantity"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


## 保存清理后的数据

In [58]:
cleaned_df.to_csv("e_commerce_cleaned.csv", index=False)